# Multi-Geometry Covariance Protocol Histories

Runs canonical GPU Markov dynamics for a small set of domain-wall lattices and saves full top-layer covariance histories for cycles `1..10`. The notebook writes one fixed campaign under `gpu_data/covariance_protocol_histories/` and records both `perfect_correction` and `postselect` protocol cases for each geometry.

This notebook uses `classA_U1FGTN_gpu.run_markov_circuit(...)` directly and does not reimplement the circuit update loop inline.


## Configuration Summary

This notebook generates one fixed `multi_geometry_C10_dwtrunc1_init-default_nsh1_S10` campaign with the following settings:

| Nx | Ny | nshell | init mode | protocols / requested samples | effective saved samples | cycles | saved history | dtype | dw truncation | trial orbital | alpha_1 / alpha_2 |
|---:|---:|---:|---|---|---|---:|---|---|---|---|---|
| 16 | 20 | `1` | `default` | `perfect_correction` / 10; `postselect` / 10 | `perfect_correction` / 10; `postselect` / 1 | 10 | cycles `1..10` | `complex128` | `True` | `X` | 1 / 30 |
| 16 | 25 | `1` | `default` | `perfect_correction` / 10; `postselect` / 10 | `perfect_correction` / 10; `postselect` / 1 | 10 | cycles `1..10` | `complex128` | `True` | `X` | 1 / 30 |
| 16 | 30 | `1` | `default` | `perfect_correction` / 10; `postselect` / 10 | `perfect_correction` / 10; `postselect` / 1 | 10 | cycles `1..10` | `complex128` | `True` | `X` | 1 / 30 |

Saved outputs:
- one compact `run_<id>` directory per protocol run
- one `manifest.json` from the canonical GPU driver per run
- one `run_summary.json` plus `latest_run.json` per run directory
- one campaign manifest plus `latest_campaign.json` for the campaign root

`postselect=True` is intentionally passed with `samples=10` so the canonical GPU driver can apply its built-in deterministic override to `samples=1`. Histories are saved with `G_history=True`, `save_init=False`, and `save_history_stride=None`, so each shard contains exactly 10 covariance matrices per sample.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
import sys

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)

BUNDLE_NAME = 'colab_small_system_testing'

def is_bundle_root(path: Path) -> bool:
    return (path / 'src' / 'classA_U1FGTN_gpu.py').exists()

candidates = []
for base in [Path.cwd(), *Path.cwd().parents]:
    candidates.extend([base, base / BUNDLE_NAME])
if IN_COLAB:
    candidates.extend([
        Path('/content/drive/MyDrive') / BUNDLE_NAME,
        Path('/content/drive/MyDrive/class_A_fermionic_adaptive_circuit') / BUNDLE_NAME,
    ])

seen = set()
BUNDLE_ROOT = None
for candidate in candidates:
    candidate = candidate.resolve()
    if candidate in seen:
        continue
    seen.add(candidate)
    if is_bundle_root(candidate):
        BUNDLE_ROOT = candidate
        break
if BUNDLE_ROOT is None:
    raise FileNotFoundError(f'Could not locate {BUNDLE_NAME} with src/classA_U1FGTN_gpu.py')

SRC_DIR = BUNDLE_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f'[setup] bundle root: {BUNDLE_ROOT}')


In [ ]:
import gc
from typing import Any

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import torch

from classA_U1FGTN_gpu import classA_U1FGTN_gpu

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is required for this Colab GPU campaign.')

torch.set_grad_enabled(False)
total_gib = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)

print(f'[setup] torch={torch.__version__}')
print(f'[setup] cuda device: {torch.cuda.get_device_name(0)}')
print(f'[setup] total gpu memory: {total_gib:.2f} GiB')
if total_gib < 35.0:
    print('[warn] This notebook is sized for an A100 40GB runtime.')


In [ ]:
GEOMETRIES = [
    {'Nx': 16, 'Ny': 20},
    {'Nx': 16, 'Ny': 25},
    {'Nx': 16, 'Ny': 30},
]
NSHELL_VALUES = [1]
INIT_MODE = 'default'
REQUESTED_SAMPLES = 10
CYCLES = 10
SAVE_INIT = False
SAVE_HISTORY_STRIDE = None
SAVED_HISTORY_LENGTH = CYCLES
ALPHA_1 = 1.0
ALPHA_2 = 30.0
DW_TRUNCATION = True
DTYPE = 'complex128'
DEVICE = 'cuda:0'
BACKEND = 'local'
SEQUENCE = 'raster_y'
BATCH_SIZE = None
TRIAL_ORBITALS = 'X'
SAVE_SUFFIX = 'history'

PROTOCOL_CONFIGS = [
    {
        'protocol': 'perfect_correction',
        'postselect': False,
        'perfect_correction': True,
        'samples_requested': REQUESTED_SAMPLES,
    },
    {
        'protocol': 'postselect',
        'postselect': True,
        'perfect_correction': False,
        'samples_requested': REQUESTED_SAMPLES,
    },
]

GPU_DATA_ROOT = BUNDLE_ROOT / 'gpu_data'
CAMPAIGN_NAME = 'covariance_protocol_histories'
OUTPUT_ROOT = GPU_DATA_ROOT / CAMPAIGN_NAME
CAMPAIGN_ID = 'multi_geometry_C10_dwtrunc1_init-default_nsh1_S10'
CAMPAIGN_ROOT = OUTPUT_ROOT / 'campaigns' / CAMPAIGN_ID
RUNS_ROOT = CAMPAIGN_ROOT / 'runs'
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

CONFIGS = [
    {
        **geometry,
        'nshell': int(nshell),
        'init_mode': INIT_MODE,
        **protocol_cfg,
    }
    for geometry in GEOMETRIES
    for nshell in NSHELL_VALUES
    for protocol_cfg in PROTOCOL_CONFIGS
]

CANONICAL_ENTRY_POINT = 'classA_U1FGTN_gpu.run_markov_circuit'

print(json.dumps({
    'campaign_id': CAMPAIGN_ID,
    'campaign_name': CAMPAIGN_NAME,
    'geometries': GEOMETRIES,
    'configs': CONFIGS,
    'cycles': CYCLES,
    'saved_history_length': SAVED_HISTORY_LENGTH,
    'dtype': DTYPE,
    'canonical_dynamics_entry_point': CANONICAL_ENTRY_POINT,
    'output_root': str(OUTPUT_ROOT),
    'campaign_root': str(CAMPAIGN_ROOT),
    'runs_root': str(RUNS_ROOT),
    'note': 'postselect=True is requested with 10 samples, and the canonical GPU driver reduces it to 1 deterministic sample.',
}, indent=2))


In [ ]:
def load_json(path: Path) -> Any:
    with Path(path).open('r', encoding='utf-8') as fh:
        return json.load(fh)


def write_json_atomic(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_suffix(path.suffix + '.tmp')
    with tmp_path.open('w', encoding='utf-8') as fh:
        json.dump(payload, fh, indent=2, sort_keys=True)
    tmp_path.replace(path)


def rel_to_gpu_data(path) -> str | None:
    if path is None:
        return None
    try:
        return str(Path(path).resolve().relative_to(GPU_DATA_ROOT.resolve()))
    except Exception:
        return None


def gib(byte_count: int | float) -> float:
    return float(byte_count) / 1024 ** 3


def mib(byte_count: int | float) -> float:
    return float(byte_count) / 1024 ** 2


def expected_raw_bytes(*, nx: int, ny: int, samples: int, history_length: int, dtype: str = 'complex128') -> int:
    nlayer = 2 * int(nx) * int(ny)
    return int(samples) * int(history_length) * nlayer * nlayer * np.dtype(dtype).itemsize


def case_output_dir(cfg: dict[str, Any]) -> Path:
    return RUNS_ROOT / f"N{cfg['Nx']}x{cfg['Ny']}_nsh{cfg['nshell']}_init-{cfg['init_mode']}_{cfg['protocol']}"


def domain_wall_metadata(model: classA_U1FGTN_gpu) -> dict[str, Any]:
    dw_loc = [int(x) for x in getattr(model, 'DW_loc', [])]
    metadata = {'dw_loc': dw_loc}
    if len(dw_loc) == 2:
        metadata['topological_x_range'] = [dw_loc[0], dw_loc[1]]
        metadata['trivial_x_segments'] = [
            [0, max(-1, dw_loc[0] - 1)],
            [min(model.Nx, dw_loc[1] + 1), model.Nx - 1],
        ]
    else:
        metadata['topological_x_range'] = None
        metadata['trivial_x_segments'] = None
    return metadata


def update_gpu_data_index() -> None:
    index_path = GPU_DATA_ROOT / 'index.json'
    if index_path.exists():
        payload = load_json(index_path)
    else:
        payload = {'bundle_root': str(BUNDLE_ROOT), 'gpu_data_root': str(GPU_DATA_ROOT), 'campaigns': []}

    campaigns = [entry for entry in payload.get('campaigns', []) if entry.get('name') != CAMPAIGN_NAME]
    campaigns.append({
        'name': CAMPAIGN_NAME,
        'kind': 'raw_covariance_histories',
        'manifest': f'{CAMPAIGN_NAME}/latest_campaign.json',
    })
    payload['bundle_root'] = str(BUNDLE_ROOT)
    payload['gpu_data_root'] = str(GPU_DATA_ROOT)
    payload['campaigns'] = campaigns
    write_json_atomic(index_path, payload)


def build_run_summary(cfg: dict[str, Any], result: dict[str, Any], model: classA_U1FGTN_gpu) -> dict[str, Any]:
    run_dir = Path(result['run_dir'])
    manifest_path = Path(result['manifest_path'])
    manifest = load_json(manifest_path)
    shards = manifest.get('shards', {})
    if sorted(int(k) for k in shards) != [0]:
        raise ValueError(f"Expected exactly one shard for {cfg['protocol']}, got {sorted(shards)}")

    shard_meta = shards['0']
    shard_path = run_dir / shard_meta['filename']
    shard = np.load(shard_path, mmap_mode='r', allow_pickle=False)
    nlayer = 2 * int(cfg['Nx']) * int(cfg['Ny'])
    expected_shape = (int(result['samples']), SAVED_HISTORY_LENGTH, nlayer, nlayer)
    if tuple(int(x) for x in shard.shape) != expected_shape:
        raise ValueError(f'{cfg["protocol"]}: expected shard shape {expected_shape}, got {shard.shape}')
    if shard.dtype != np.complex128:
        raise ValueError(f'{cfg["protocol"]}: expected complex128 shard, got {shard.dtype}')
    if manifest.get('store_mode') != 'history':
        raise ValueError(f'{cfg["protocol"]}: expected history store mode in manifest')

    raw_saved_bytes = expected_raw_bytes(
        nx=cfg['Nx'],
        ny=cfg['Ny'],
        samples=int(result['samples']),
        history_length=SAVED_HISTORY_LENGTH,
        dtype='complex128',
    )
    raw_requested_bytes = expected_raw_bytes(
        nx=cfg['Nx'],
        ny=cfg['Ny'],
        samples=int(cfg['samples_requested']),
        history_length=SAVED_HISTORY_LENGTH,
        dtype='complex128',
    )
    actual_file_bytes = shard_path.stat().st_size
    header_bytes = int(actual_file_bytes - raw_saved_bytes)
    if header_bytes < 0:
        raise ValueError(f'{cfg["protocol"]}: file size smaller than theoretical raw payload')

    dw_meta = domain_wall_metadata(model)
    summary = {
        'canonical_dynamics_entry_point': CANONICAL_ENTRY_POINT,
        'campaign_id': CAMPAIGN_ID,
        'campaign_name': CAMPAIGN_NAME,
        'Nx': int(cfg['Nx']),
        'Ny': int(cfg['Ny']),
        'nshell': int(cfg['nshell']),
        'cycles': CYCLES,
        'history_length_saved': SAVED_HISTORY_LENGTH,
        'save_init': SAVE_INIT,
        'save_history_stride': SAVE_HISTORY_STRIDE,
        'protocol': cfg['protocol'],
        'postselect': bool(cfg['postselect']),
        'perfect_correction': bool(cfg['perfect_correction']),
        'samples_requested': int(cfg['samples_requested']),
        'samples': int(result['samples']),
        'init_mode': cfg['init_mode'],
        'alpha_1': ALPHA_1,
        'alpha_2': ALPHA_2,
        'dw_truncation': DW_TRUNCATION,
        'trial_orbitals': TRIAL_ORBITALS,
        'dtype_requested': DTYPE,
        'dtype_resolved': 'complex128',
        'device': DEVICE,
        'backend_requested': BACKEND,
        'backend_resolved': str(model.backend),
        'sequence': SEQUENCE,
        'batch_size_requested': BATCH_SIZE,
        'batch_size': int(result['batch_size']),
        'batch_size_mode': result['batch_size_mode'],
        'batch_size_auto_info': result.get('batch_size_auto_info'),
        'manifest_store_mode': manifest.get('store_mode'),
        'manifest_num_batches': int(manifest.get('num_batches', -1)),
        'snapshot_cycles': None,
        'shard_shape': list(expected_shape),
        'shard_dtype': str(shard.dtype),
        'shard_file_name': shard_meta['filename'],
        'shard_path': str(shard_path),
        'shard_path_relative': rel_to_gpu_data(shard_path),
        'shard_file_size_bytes': int(actual_file_bytes),
        'estimated_requested_history_storage_bytes': int(raw_requested_bytes),
        'estimated_requested_history_storage_gib': round(gib(raw_requested_bytes), 4),
        'estimated_saved_history_storage_bytes': int(raw_saved_bytes),
        'estimated_saved_history_storage_gib': round(gib(raw_saved_bytes), 4),
        'npy_header_overhead_bytes': header_bytes,
        'run_dir': str(run_dir),
        'manifest_path': str(manifest_path),
        'save_path': str(result['save_path']),
        'run_dir_relative': rel_to_gpu_data(run_dir),
        'manifest_path_relative': rel_to_gpu_data(manifest_path),
        'save_path_relative': rel_to_gpu_data(result['save_path']),
        'dw_loc': dw_meta['dw_loc'],
        'topological_x_range': dw_meta['topological_x_range'],
        'trivial_x_segments': dw_meta['trivial_x_segments'],
    }
    del shard
    return summary


In [ ]:
def run_one_config(cfg: dict[str, Any]) -> dict[str, Any]:
    out_dir = case_output_dir(cfg)
    out_dir.mkdir(parents=True, exist_ok=True)
    print(
        f"[config] Nx={cfg['Nx']} Ny={cfg['Ny']} nshell={cfg['nshell']} protocol={cfg['protocol']} "
        f"requested_samples={cfg['samples_requested']} cycles={CYCLES} output={out_dir}"
    )

    model = classA_U1FGTN_gpu(
        Nx=int(cfg['Nx']),
        Ny=int(cfg['Ny']),
        DW=True,
        nshell=int(cfg['nshell']),
        filling_frac=0.5,
        alpha_1=ALPHA_1,
        alpha_2=ALPHA_2,
        trial_orbitals=TRIAL_ORBITALS,
        dw_truncation=DW_TRUNCATION,
        triv_region_local_mode=False,
        device=DEVICE,
        dtype=DTYPE,
        backend=BACKEND,
    )

    result = model.run_markov_circuit(
        G_history=True,
        progress=False,
        cycles=CYCLES,
        postselect=bool(cfg['postselect']),
        perfect_correction=bool(cfg['perfect_correction']),
        samples=int(cfg['samples_requested']),
        init_mode=cfg['init_mode'],
        save=True,
        save_init=SAVE_INIT,
        save_history_stride=SAVE_HISTORY_STRIDE,
        snapshot_cycles=None,
        save_suffix=SAVE_SUFFIX,
        n_a=0.5,
        sequence=SEQUENCE,
        batch_size=BATCH_SIZE,
        save_dir=str(out_dir),
        resume=True,
        merge_on_finish=False,
        return_data=False,
        run_dir_style='compact',
    )

    summary = build_run_summary(cfg, result, model)
    summary_path = Path(result['run_dir']) / 'run_summary.json'
    summary['summary_path'] = str(summary_path)
    summary['summary_path_relative'] = rel_to_gpu_data(summary_path)
    write_json_atomic(summary_path, summary)

    latest_path = out_dir / 'latest_run.json'
    write_json_atomic(
        latest_path,
        {
            'run_summary_path': str(summary_path),
            'run_summary_path_relative': rel_to_gpu_data(summary_path),
            **summary,
        },
    )

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return summary


In [ ]:
all_results = []
for cfg in tqdm(CONFIGS, desc='Configurations', unit='config'):
    all_results.append(run_one_config(cfg))

results_by_geometry = {}
for row in all_results:
    key = f"N{row['Nx']}x{row['Ny']}"
    results_by_geometry.setdefault(key, {})[row['protocol']] = {
        'dw_loc': row['dw_loc'],
        'topological_x_range': row['topological_x_range'],
        'trivial_x_segments': row['trivial_x_segments'],
    }

campaign_manifest = {
    'canonical_dynamics_entry_point': CANONICAL_ENTRY_POINT,
    'bundle_root': str(BUNDLE_ROOT),
    'gpu_data_root': str(GPU_DATA_ROOT),
    'output_root': str(OUTPUT_ROOT),
    'campaign_root': str(CAMPAIGN_ROOT),
    'runs_root': str(RUNS_ROOT),
    'output_root_relative': rel_to_gpu_data(OUTPUT_ROOT),
    'campaign_root_relative': rel_to_gpu_data(CAMPAIGN_ROOT),
    'runs_root_relative': rel_to_gpu_data(RUNS_ROOT),
    'campaign_name': CAMPAIGN_NAME,
    'campaign_id': CAMPAIGN_ID,
    'geometries': GEOMETRIES,
    'configs': CONFIGS,
    'results': all_results,
    'requested_samples': REQUESTED_SAMPLES,
    'cycles': CYCLES,
    'store_mode': 'history',
    'history_length_saved': SAVED_HISTORY_LENGTH,
    'save_init': SAVE_INIT,
    'save_history_stride': SAVE_HISTORY_STRIDE,
    'snapshot_cycles': None,
    'dtype': DTYPE,
    'init_mode': INIT_MODE,
    'dw_truncation': DW_TRUNCATION,
    'trial_orbitals': TRIAL_ORBITALS,
    'backend': BACKEND,
    'sequence': SEQUENCE,
    'domain_wall_reference_by_geometry': results_by_geometry,
    'note': 'postselect=True was requested with 10 samples and saved with 1 canonical deterministic sample.',
}
campaign_manifest_path = CAMPAIGN_ROOT / 'campaign_manifest.json'
write_json_atomic(campaign_manifest_path, campaign_manifest)

latest_campaign_path = OUTPUT_ROOT / 'latest_campaign.json'
write_json_atomic(
    latest_campaign_path,
    {
        'campaign_name': CAMPAIGN_NAME,
        'campaign_id': CAMPAIGN_ID,
        'manifest': rel_to_gpu_data(campaign_manifest_path),
        'campaign_manifest_path': str(campaign_manifest_path),
        'campaign_manifest_path_relative': rel_to_gpu_data(campaign_manifest_path),
    },
)

update_gpu_data_index()
print(f'[done] campaign manifest: {campaign_manifest_path}')
print(f'[done] latest campaign pointer: {latest_campaign_path}')


In [ ]:
summary_df = pd.DataFrame(
    [
        {
            'Nx': row['Nx'],
            'Ny': row['Ny'],
            'protocol': row['protocol'],
            'samples_requested': row['samples_requested'],
            'samples_saved': row['samples'],
            'batch_size': row['batch_size'],
            'batch_size_mode': row['batch_size_mode'],
            'estimated_saved_gib': row['estimated_saved_history_storage_gib'],
            'shard_file_gib': round(gib(row['shard_file_size_bytes']), 4),
            'dw_loc': row['dw_loc'],
            'run_dir_relative': row['run_dir_relative'],
        }
        for row in all_results
    ]
).sort_values(['Nx', 'Ny', 'protocol']).reset_index(drop=True)
summary_df


In [ ]:
if IN_COLAB:
    from google.colab import runtime  # type: ignore
    runtime.unassign()
else:
    print('[done] not running in Colab; runtime disconnect skipped')
